## Deploy zero-shot-classification models from HuggingFaceHub to AzureML Online Endpoints

Notebook from azureml-examples repo, original path:
Users/niswitan/azureml-examples/sdk/python/foundation-models/huggingface/inference/zero-shot-classification/zero-shot-classification-online-endpoint.ipynb

This sample shows how to deploy `zero-shot-classification` models from the HuggingFaceHub to an online endpoint for inference. Learn more about `zero-shot-classification` task: https://huggingface.co/tasks/zero-shot-classification

A large set of models hosted on [Hugging Face Hub](https://huggingface.co/models) are available in the Hugging Face Hub collection in AzureML Model Catalog. This collection is powered by the Hugging Face Hub community registry. Integration with the AzureML Model Catalog enables seamless deployment of Hugging Face Hub models in AzureML. _todo: learn more link_

### Outline
* Set up pre-requisites.
* Pick a model to deploy.
* Deploy the model for real time inference.
* Try sample inference.
* Clean up resources.

### Set up pre-requisites
* Install dependencies
* Connect to AzureML Workspace. Learn more at [set up SDK authentication](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-setup-authentication?tabs=sdk). Replace  `<WORKSPACE_NAME>`, `<RESOURCE_GROUP>` and `<SUBSCRIPTION_ID>` below.
* Connect to `HuggingFaceHub` community registry

In [21]:
from azure.ai.ml import MLClient
from azure.identity import (
    DefaultAzureCredential,
    InteractiveBrowserCredential,
    ClientSecretCredential,
)
from azure.ai.ml.entities import AmlCompute
import time

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    credential = InteractiveBrowserCredential()

# connect to a workspace
workspace_ml_client = None
try:
    workspace_ml_client = MLClient.from_config(credential)
    subscription_id = workspace_ml_client.subscription_id
    workspace = workspace_ml_client.workspace_name
    resource_group = workspace_ml_client.resource_group_name
except Exception as ex:
    print(ex)
    # Enter details of your workspace
    subscription_id = "<SUBSCRIPTION_ID>"
    resource_group = "<RESOURCS_GROUP>"
    workspace = "<WORKSPACE_NAME>"
    workspace_ml_client = MLClient(
        credential, subscription_id, resource_group, workspace
    )
# Connect to the HuggingFaceHub registry
registry_ml_client = MLClient(credential, registry_name="HuggingFace")
print(registry_ml_client)

Found the config file in: /config.json


MLClient(credential=<azure.identity._credentials.default.DefaultAzureCredential object at 0x7f0f921049a0>,
         subscription_id=d5210851-d1ca-44ac-8071-a0c7191e9631,
         resource_group_name=prod-azure-ml-registry,
         workspace_name=None)


### Pick a model to deploy

Open the Model Catalog in AzureML Studio and choose the Hugging Face Hub collection. Filter by the `zero-shot-classification` task and search any specific models you are interested in. In this example, we use the `facebook-bart-large-mnli` model. If you plan to deploy a different model, replace the model name and version accordingly. 

In [19]:
model_name = "facebook-bart-large-mnli"
foundation_model = registry_ml_client.models.get(model_name, version="19")
print(
    "\n\nUsing model name: {0}, version: {1}, id: {2} for inferencing".format(
        foundation_model.name, foundation_model.version, foundation_model.id
    )
)



Using model name: facebook-bart-large-mnli, version: 19, id: azureml://registries/HuggingFace/models/facebook-bart-large-mnli/versions/19 for inferencing


### Deploy the model to an online endpoint
Online endpoints give a durable REST API that can be used to integrate with applications that need to use the model. Create an online endpoint and then create an online deployment. You need to specify the Virtual Machine instance or SKU when creating the deployment. You can find the optimal CPU or GPU SKU for a model by opening the quick deployment dialog from the model page in the AzureML Model Catalog. Specify the SKU in the `instance_type` input in deployment settings below.

Typically Online Endpoints require you to provide scoring script and a docker container image (through an AzureML environment), in addition to the model. You don't need to worry about them for HuggingFace Hub models available in AzureML Model Catalog because we have enabled 'no code deployments' for these models by packaging scoring script and container image along with the model.

Learn more about Online Endpoints: https://learn.microsoft.com/en-us/azure/machine-learning/how-to-deploy-online-endpoints

In [3]:
import time, sys
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    OnlineRequestSettings,
)

# Create online endpoint - endpoint names need to be unique in a region, hence using timestamp to create unique endpoint name
timestamp = int(time.time())
online_endpoint_name = "zero-shot-cla-" + str(timestamp)
# create an online endpoint
endpoint = ManagedOnlineEndpoint(
    name=online_endpoint_name,
    description="Online endpoint for "
    + foundation_model.name
    + ", for zero-shot-classification task",
    auth_mode="key",
    # public_network_access="disabled"
)
workspace_ml_client.begin_create_or_update(endpoint).wait()

In [20]:
# create a deployment
demo_deployment = ManagedOnlineDeployment(
    name="demo",
    endpoint_name=online_endpoint_name,
    model=foundation_model.id,
    instance_type="Standard_DS3_v2",
    instance_count=1,
)
workspace_ml_client.online_deployments.begin_create_or_update(demo_deployment).wait()
# online endpoints can have multiple deployments with traffic split or shadow traffic. Set traffic to 100% for demo deployment
endpoint.traffic = {"demo": 100}
workspace_ml_client.begin_create_or_update(endpoint).result()

Check: endpoint zero-shot-cla-1712862242 exists


..........................................................................................

### Try sample inference

Online endpoints expose a REST API that can be integrated into your applications. Learn how to fetch the scoring REST API and credentials for online endpoints here: https://learn.microsoft.com/en-us/azure/machine-learning/how-to-authenticate-online-endpoint

In this example, we will use the Python SDK helper method to invoke the endpoint. 

In [5]:
# # Get the model object from HuggingFaceHub. We can use it to check for sample test data
# import urllib.request, json

# raw_data = urllib.request.urlopen(
#     "https://huggingface.co/api/models/" + foundation_model.tags["modelId"]
# )

# print("https://huggingface.co/api/models/" + foundation_model.tags["modelId"])
# data = json.load(raw_data)
# print(data)

In [6]:
# # check if there is sample inference data available on HuggingFaceHub for the model, else try with the backup sample data
# scoring_file = "./sample_score.json"
# score_dict = {}
# input_sequence = []
# candidate_label = []
# if "widgetData" in data:
#     input = data["widgetData"][0]
#     # input_sequence.append(input["text"])
#     # candidate_label.append(input["candidate_labels"])
#     # write the sample_score.json file
#     score_dict["inputs"] = input["text"]
#     score_dict["parameters"] = {"candidate_labels": input["candidate_labels"]}
#     with open(scoring_file, "w") as outfile:
#         json.dump(score_dict, outfile, indent=2)
# else:
#     scoring_file = "./sample_score_backup.json"

# # print the sample scoring file
# print("\n\nSample scoring file: ")
# with open(scoring_file) as json_file:
#     scoring_data = json.load(json_file)
#     print(scoring_data)

In [7]:
# import csv, json

# data_jsonl = []
# score_dict = {}
# candidate_labels = "thrust bearings issue, pump seal leak, broken breaker, pipe leak, check valve broken"
# multi_class = False
# row_to_keep = 12

# with open('data/sample_data.csv', mode='r') as csv_file:
# #    csv_reader = csv.DictReader(csv_file)
#     csv_reader = csv.reader(csv_file)
#     line_count = 0
#     for row in csv_reader:
#         if line_count == 0:
#             print(f'Column names are {", ".join(row)}')
# #        print(f'\t{row["\ufeffShort Description"]}')
#         # data_jsonl.append(row)
#         elif line_count == row_to_keep:   # only keep single row
#             wo_long_text = row[1] # long text description
#             score_dict["inputs"] = wo_long_text 
#             score_dict["parameters"] = {
#                 "candidate_labels": candidate_labels, 
#                 "multi_class": multi_class}
#         else: 
#             pass
#         line_count += 1

# print("\nscore_dict: ")
# print(score_dict)

# scoring_file = "data/sample_score.json"
# with open(scoring_file, "w") as outfile:
#         json.dump(score_dict, outfile, indent=2)

# # print the sample scoring file
# print("\n\nSample scoring file: ")
# with open(scoring_file) as json_file:
#     scoring_data = json.load(json_file)
#     print(scoring_data)

In [8]:
# online_endpoint_name = "zero-shot-cla-1712229133"
# # score the sample_score.json file using the online endpoint with the azureml endpoint invoke method
# response = workspace_ml_client.online_endpoints.invoke(
#     endpoint_name=online_endpoint_name,
#     deployment_name="demo",
#     request_file=scoring_file,
# )
# response_json = json.loads(response)
# print(json.dumps(response_json, indent=2))

In [9]:
# label, score = response_json['labels'][0], response_json['scores'][0], 
# print(f'{label} ({score:0.3f})')

In [10]:
# json.dumps(score_dict, indent=2)

In [22]:
import json

def construct_zero_shot_request_body(note_text, candidate_labels):

    data = {
        "inputs": note_text,
        "parameters": {
            "candidate_labels": candidate_labels,
            "multi_class": False
            }
        }

    body = str.encode(json.dumps(data))
    
    return body

In [23]:
note_text = "Contacted a patient to discuss payment options for a high deductible plan balance." # sample note text
candidate_labels = "Patient Demographics,\
Insurance Information,\
Provider Information,\
Verify Insurance Eligibility and Benefits,\
Coordination of Benefits (COB),\
Obtain Prior Authorizations,\
Medical Records,\
Itemized Bill or Charge Sheet,\
Authorization and Referral Information,\
Advance Beneficiary Notice (ABN),\
Accurate Coding,\
Claim Submission,\
Follow-up on Submitted Claims,\
Denial Management,\
Payment Posting,\
Patient Billing,\
Reconciliation,\
Reporting"
testbody = construct_zero_shot_request_body(note_text, candidate_labels)
print(testbody)

b'{"inputs": "Contacted a patient to discuss payment options for a high deductible plan balance.", "parameters": {"candidate_labels": "Patient Demographics,Insurance Information,Provider Information,Verify Insurance Eligibility and Benefits,Coordination of Benefits (COB),Obtain Prior Authorizations,Medical Records,Itemized Bill or Charge Sheet,Authorization and Referral Information,Advance Beneficiary Notice (ABN),Accurate Coding,Claim Submission,Follow-up on Submitted Claims,Denial Management,Payment Posting,Patient Billing,Reconciliation,Reporting", "multi_class": false}}'


In [24]:
url = "https://zero-shot-cla-1712862242.eastus.inference.ml.azure.com/score"
api_key = "tgcxwIIxZsnDZrapcSY1jzlgT8V2oelh"
headers = {
    'Content-Type':'application/json', 
    'Authorization':('Bearer '+ api_key), 
    'azureml-model-deployment': 'facebook-bart-large-mnli-19' 
    }

In [25]:
import urllib.request
req = urllib.request.Request(url, testbody, headers)
response = urllib.request.urlopen(req, timeout=10)
result = response.read()
print(result)

HTTPError: HTTP Error 404: Not Found

In [57]:
label, score = json.loads(result)['labels'][0], response_json['scores'][0], 
print(f'{label} ({score:0.3f})')

thrust bearings issue (0.961)


In [62]:
import pandas as pd
import csv

results = []

with open('data/sample_data.csv', mode='r') as csv_file:
    csv_reader = csv.reader(csv_file)
    # line_count = 0
    for i, row in enumerate(csv_reader):
        if i == 0:
            print(f'Column names are {", ".join(row)}')
        # elif i < 10:   # uncomment when testing to restrict to subset of rows
        else:           # use when iterating over all rows
            print(i)
            note_id = row[0]            # note id
            note_text = row[1]          # note text
            action_category = row[2]    # actual category
            
            # construct request
            body = construct_zero_shot_request_body(note_text, candidate_labels)
            req = urllib.request.Request(url, body, headers)

            # call model API
            try:
                response = urllib.request.urlopen(req, timeout=10)

                result = json.loads(response.read())
                predicted_category, score = result['labels'][0], result['scores'][0]
                # collect results in row
                results.append((note_id, note_text, action_category, predicted_category, score))

            except urllib.error.HTTPError as error:
                print("The request failed with status code: " + str(error.code))

                # Print the headers - they include the requert ID and the timestamp, which are useful for debugging the failure
                print(error.info())
                print(error.read().decode("utf8", 'ignore'))
        # else:                 # uncomment when testing
            # pass              # uncomment when testing

df = pd.DataFrame(results, columns=["note_id", "note_text", "action_category", "predicted_category", "score"])


Column names are ﻿Short Description, Long Text Description, Failure Mode Category  
1
2
3
4
5
6
7
8
9


In [63]:
df

,WO_Desc,failure_mode_actual,failure_mode_predicted,score
0,P-5270A high vibrations,thrust bearings issue,thrust bearings issue,0.966454
1,P-6336A MVR W-438 pump seal leak,pump seal leak,pump seal leak,0.989889
2,P-5470(D) OIL LEAK ON PIPING TO BEARINGS,pipe leak,pipe leak,0.883303
3,BKR-221D tripping intermittently,broken breaker,broken breaker,0.982678
4,T-1003A steam leak detected,pipe leak,pipe leak,0.974656
5,CHK-5573 not closing properly,check valve broken,check valve broken,0.969013
6,V-4022 abnormal noise,thrust bearings issue,thrust bearings issue,0.977483
7,P-2105B seal failure observed,pump seal leak,pump seal leak,0.983039
8,Pipe L-1010 corrosion hole,pipe leak,pipe leak,0.973194


Calculate accuracy.

In [ ]:
# calculate accuracy in the df


#### Construct confusion matrix

In [ ]:
# create and display confusion matrix

### Delete the online endpoint
Don't forget to delete the online endpoint, else you will leave the billing meter running for the compute used by the endpoint.

In [37]:
workspace_ml_client.online_endpoints.begin_delete(name=online_endpoint_name).wait()

.......................................................................